#  Kiểm thử toàn diện Package `qc` (Data Quality Control)

Notebook này kiểm thử và trực quan hóa 4 hàm nòng cốt của package `qc`:
1. `plot_missing`: Tỷ lệ giá trị thiếu theo cột.
2. `plot_duplicate`: Tỷ lệ dòng trùng theo một hoặc nhiều định nghĩa.
3. `plot_outlier_rate`: Tỷ lệ ngoại lai theo cột số (IQR, Z-score, MAD).
4. `plot_entity_coverage`: Độ phủ thực thể (khách hàng, sản phẩm...) theo thời gian.

---
###  Quy ước thiết kế của package `qc`:
- **Nguyên tắc**: Mỗi hàm tự tính, tự vẽ và trả về `(Figure, DataFrame)` với số liệu chi tiết đằng sau chart.
- **Tỷ lệ**: Chuẩn hóa trong khoảng `[0, 1]`.
- **Lọc và Cảnh báo**:
  - `min_*`: Lọc bớt các cột có tỷ lệ nhỏ khỏi cả bảng lẫn biểu đồ.
  - `threshold`: Tô đỏ phần vượt ngưỡng (hoặc sụt dưới ngưỡng với coverage) và vẽ đường đứt nét cảnh báo; bảng trả về tự động có thêm cột boolean `exceed`.
- **Linh hoạt**: Nhận `ax` để dễ dàng ghép vào subplot / dashboard phức hợp; không tự ý gọi `plt.show()` hay sửa đổi `df` gốc.



In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Thiết lập đường dẫn import package qc và data generator
repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from qc import (
    plot_missing,
    plot_duplicate,
    plot_outlier_rate,
    plot_entity_coverage,
)
from data.generate_mock_data import generate_qc_mock_data

# Tùy chỉnh hiển thị
%matplotlib inline
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

print(" Đã import thành công package qc!")



## 0. Tải dữ liệu giả lập (Mock Dataset)

Dữ liệu giả lập mô phỏng giao dịch thương mại điện tử với:
- 5,270 dòng, 17 cột đa dạng kiểu dữ liệu (`datetime`, `float64`, `int64`, `string`, `object`).
- Có sẵn các mức độ thiếu dữ liệu từ 0% đến 100%.
- Có dòng trùng toàn bộ, trùng theo mã đơn, và trùng theo tổ hợp `(customer_id, order_date)`.
- Có ngoại lai ở nhiều biến số, kèm trường hợp biên như cột hằng số (`std=0`) và giá trị vô cùng (`±inf`).
- Có 300 khách hàng trong 18 tháng, gồm tháng bình thường, tháng cao điểm, tháng tụt giảm mạnh và tháng gián đoạn.



In [ ]:
csv_file = repo_root / "data" / "mock_transactions.csv"
if csv_file.exists():
    df = pd.read_csv(csv_file, parse_dates=["order_date"])
else:
    df = generate_qc_mock_data(n_rows=5000, seed=42)

print(f"Kích thước dataset: {df.shape[0]:,} dòng x {df.shape[1]} cột\n")
display(df.head(5))
display(df.dtypes.to_frame(name="Dtype"))



## 1. Kiểm thử hàm `plot_missing` (Tỷ lệ thiếu theo cột)

Hàm tính tỷ lệ `NaN / None` của các cột và vẽ bar nằm ngang sắp xếp theo thứ tự giảm dần.

### 1.1. Chạy mặc định trên tất cả các cột (`cols=None`)



In [ ]:
fig, table = plot_missing(df)
display(table)



### 1.2. Chọn cột cụ thể (`cols`) và loại trừ (`exclude`)



In [ ]:
# Chỉ kiểm tra một nhóm cột kinh doanh, bỏ qua ID
selected_cols = ["customer_id", "total_amount", "customer_age", "payment_method", "shipping_address", "customer_feedback"]
fig, table = plot_missing(df, cols=selected_cols, exclude=["customer_id"])
display(table)



### 1.3. Lọc cột có tỷ lệ thiếu đáng kể với `min_missing`
Chỉ hiển thị các cột có `missing_rate >= 0.10` (10% trở lên).



In [ ]:
fig, table = plot_missing(df, min_missing=0.10)
display(table)



### 1.4. Thiết lập ngưỡng cảnh báo với `threshold`
Cột có tỷ lệ thiếu `> threshold` (ví dụ 30%) sẽ được **tô đỏ** và biểu đồ vẽ một đường thẳng đứt nét làm mốc.
Bảng trả về có thêm cột bool `exceed`.



In [ ]:
fig, table = plot_missing(df, threshold=0.30, title="Cảnh báo: Các cột có tỷ lệ thiếu > 30%")
display(table)



---
## 2. Kiểm thử hàm `plot_duplicate` (Tỷ lệ dòng trùng lặp)

Hàm kiểm tra tỷ lệ trùng lặp theo một hoặc nhiều định nghĩa trùng (subset).

### 2.1. Trùng lặp toàn bộ dòng (`subset=None`)



In [ ]:
fig, table = plot_duplicate(df, subset=None, keep="first")
display(table)



### 2.2. Trùng lặp theo một khóa định danh duy nhất (`subset=['order_id']`)



In [ ]:
fig, table = plot_duplicate(df, subset=["order_id"], threshold=0.03)
display(table)



### 2.3. So sánh nhiều định nghĩa trùng lặp cùng lúc (truyền dict `subset`)
Điểm mạnh của hàm là cho phép truyền dictionary `{tên_định_nghĩa: cột_hoặc_danh_sách_cột}` để so sánh trực quan các mức độ trùng lặp.



In [ ]:
duplicate_definitions = {
    "Trùng toàn bộ dòng": list(df.columns),
    "Trùng mã đơn (order_id)": ["order_id"],
    "Trùng khách + ngày": ["customer_id", "order_date"],
    "Trùng khách + địa chỉ": ["customer_id", "shipping_address"],
    "Trùng khách + mã đơn": ["customer_id", "order_id"],
}

fig, table = plot_duplicate(
    df,
    subset=duplicate_definitions,
    threshold=0.04,
    title="So sánh các định nghĩa dòng trùng lặp (Ngưỡng cảnh báo: 4%)"
)
display(table)



### 2.4. So sánh các chiến lược `keep`
- `keep="first"`: Đánh dấu trùng từ lần xuất hiện thứ 2 trở đi.
- `keep=False`: Đánh dấu trùng toàn bộ các dòng thuộc nhóm trùng (kể cả bản ghi đầu tiên).



In [ ]:
fig1, tab1 = plot_duplicate(df, subset=["order_id"], keep="first", title="keep='first'")
fig2, tab2 = plot_duplicate(df, subset=["order_id"], keep=False, title="keep=False (tính cả bản gốc)")

display(pd.concat([
    tab1.assign(keep="first"),
    tab2.assign(keep="False")
], ignore_index=True))



---
## 3. Kiểm thử hàm `plot_outlier_rate` (Tỷ lệ ngoại lai cột số)

Hàm hỗ trợ 3 phương pháp phát hiện ngoại lai:
- `"iqr"`: Ngoài khoảng $[Q1 - k \cdot IQR, Q3 + k \cdot IQR]$ (mặc định $k=1.5$).
- `"zscore"`: Ngoài khoảng $[\mu - k \cdot \sigma, \mu + k \cdot \sigma]$ (mặc định $k=3.0$).
- `"mad"`: Modified Z-score ngoài khoảng $[\text{median} - k \cdot MAD / 0.6745, \text{median} + k \cdot MAD / 0.6745]$ (mặc định $k=3.5$).

Tự động bỏ qua giá trị thiếu và giá trị vô cùng `±inf`. Xử lý an toàn trường hợp $std=0$ hoặc $MAD=0$.

### 3.1. Phương pháp mặc định IQR (`method="iqr"`)



In [ ]:
fig, table = plot_outlier_rate(df, method="iqr")
display(table)



### 3.2. So sánh 3 phương pháp: IQR vs Z-score vs MAD



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

_, t_iqr = plot_outlier_rate(df, method="iqr", k=1.5, ax=axes[0], title="IQR (k=1.5)")
_, t_zsc = plot_outlier_rate(df, method="zscore", k=3.0, ax=axes[1], title="Z-score (k=3.0)")
_, t_mad = plot_outlier_rate(df, method="mad", k=3.5, ax=axes[2], title="MAD (k=3.5)")

plt.tight_layout()

# Ghép so sánh tỷ lệ giữa 3 phương pháp
comparison = t_iqr[["column", "outlier_rate"]].rename(columns={"outlier_rate": "rate_iqr"})
comparison["rate_zscore"] = t_zsc.set_index("column").loc[comparison["column"], "outlier_rate"].values
comparison["rate_mad"] = t_mad.set_index("column").loc[comparison["column"], "outlier_rate"].values
display(comparison)



### 3.3. Lọc `min_rate` và cảnh báo `threshold`



In [ ]:
fig, table = plot_outlier_rate(
    df,
    method="iqr",
    min_rate=0.015,
    threshold=0.03,
    title="Các cột có tỷ lệ Outlier >= 1.5% (Tô đỏ cột > 3%)"
)
display(table)



### 3.4. Kiểm tra xử lý an toàn: Cột hằng số (`std=0`) và cột có giá trị `±inf`
- `constant_fee`: Hằng số 15,000 -> $std=0, MAD=0$, hàm trả về lower=NaN, upper=NaN, n_outlier=0 (không bị văng lỗi chia cho 0).
- `metric_with_inf`: Chứa giá trị vô cùng `±inf`, hàm tự động loại bỏ giá trị vô cùng trước khi tính toán.



In [ ]:
fig, table = plot_outlier_rate(df, cols=["constant_fee", "metric_with_inf"], method="zscore")
display(table)



---
## 4. Kiểm thử hàm `plot_entity_coverage` (Độ phủ thực thể theo thời gian)

Hàm theo dõi số lượng thực thể (ví dụ: `customer_id`) có giao dịch trong từng kỳ chia cho tổng số thực thể duy nhất trong toàn bộ dữ liệu.

### 4.1. Độ phủ khách hàng theo tháng (`freq="M"`) kèm ngưỡng cảnh báo `threshold=0.30`
Tháng nào có coverage `< 30%` sẽ được đánh dấu chấm đỏ và đường đứt nét làm mốc.



In [ ]:
fig, table = plot_entity_coverage(
    df,
    entity="customer_id",
    time="order_date",
    freq="M",
    threshold=0.30,
    title="Độ phủ khách hàng theo tháng (Cảnh báo: Coverage < 30%)"
)
display(table)



### 4.2. Kiểm tra tính năng tự lấp đầy chu kỳ gián đoạn (`fill_gaps`)
Trong dữ liệu giả lập, tháng **2024-08** cố tình không có bất kỳ giao dịch nào.
Hàm `plot_entity_coverage` tự động tạo chu kỳ liên tục từ tháng nhỏ nhất đến lớn nhất (`pd.period_range`) và gán `n_entity=0`, `coverage=0.0`.



In [ ]:
# Lọc xem tháng 2024-08 trong bảng kết quả
gap_month = table[table["period"].astype(str) == "2024-08"]
display(gap_month)
assert gap_month.iloc[0]["n_entity"] == 0, "Lỗi: Kỳ trống không được gán 0!"
assert gap_month.iloc[0]["coverage"] == 0.0, "Lỗi: Coverage của kỳ trống phải bằng 0!"
print(" Xác nhận fill_gaps hoạt động chính xác: 2024-08 có n_entity = 0 và coverage = 0.0")



### 4.3. Độ phủ theo tuần (`freq="W"`) và theo ngày (`freq="D"`)



In [ ]:
fig_w, tab_w = plot_entity_coverage(df, entity="customer_id", time="order_date", freq="W", title="Độ phủ theo tuần (freq='W')")
display(tab_w.head(10))



---
## 5. Ghép Dashboard QC tổng hợp (4 trong 1)

Nhờ hỗ trợ tham số `ax`, chúng ta có thể kết hợp cả 4 biểu đồ vào một bảng điều khiển chất lượng dữ liệu hoàn chỉnh chỉ với một Figure duy nhất!



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Tỷ lệ Missing
plot_missing(
    df,
    threshold=0.30,
    ax=axes[0, 0],
    title="1. Missing Values (Ngưỡng 30%)"
)

# 2. Tỷ lệ Trùng lặp
dup_defs = {
    "Toàn bộ dòng": list(df.columns),
    "Trùng order_id": ["order_id"],
    "Trùng (khách, ngày)": ["customer_id", "order_date"],
    "Trùng (khách, địa chỉ)": ["customer_id", "shipping_address"],
}
plot_duplicate(
    df,
    subset=dup_defs,
    threshold=0.04,
    ax=axes[0, 1],
    title="2. Duplicate Rate (Ngưỡng 4%)"
)

# 3. Tỷ lệ Outlier
plot_outlier_rate(
    df,
    method="iqr",
    threshold=0.03,
    ax=axes[1, 0],
    title="3. Outlier Rate - IQR (Ngưỡng 3%)"
)

# 4. Độ phủ Entity
plot_entity_coverage(
    df,
    entity="customer_id",
    time="order_date",
    freq="M",
    threshold=0.30,
    ax=axes[1, 1],
    title="4. Customer Coverage hàng tháng (Ngưỡng 30%)"
)

fig.suptitle(" BẢNG ĐIỀU KHIỂN CHẤT LƯỢNG DỮ LIỆU TỔNG HỢP (QC DASHBOARD)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(" Hoàn thành toàn bộ quy trình kiểm thử package qc!")

